#**Query 2 Doc (Q2D)**

#START

## Cell 1: Environment, Setup, and Data Loading

In [ ]:
# 1. SETUP ENVIRONMENT
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

# 2. INSTALL TOOLS
!pip install pyserini trectools tabulate google-generativeai tqdm -q

# 3. IMPORTS & MOUNTING
from google.colab import drive, userdata
import pandas as pd
import json
import time
from tqdm import tqdm
from google.api_core import exceptions
from pyserini.search.lucene import LuceneSearcher
from trectools import TrecQrel, TrecRun, TrecEval

drive.mount('/content/drive')

# 4. CONFIGURATION PATHS
BASE_PATH = '/content/drive/MyDrive/TEXT/FinalProject'
QUERIES_PATH = os.path.join(BASE_PATH, 'queriesROBUST.txt')
QRELS_PATH = os.path.join(BASE_PATH, 'qrels_50_Queries')
#MASTER_DATA_PATH = os.path.join(BASE_PATH, 'q2d_rerank_master.json')
MASTER_DATA_PATH = os.path.join(BASE_PATH, 'q2d_part1.json') #TODO:change to "part1" for  Q (0-10) Q(10-20)Q (20-30)Q (30-40) Q(40-50)

# 5. GLOBAL RESEARCH PARAMETERS
NUM_TERMS = 15     # Number of entities to expand
NUM_SNIPPETS = 10   # Number of fragments to generate
BEST_K1 = 0.6      # Your tuned BM25 parameter
BEST_B = 0.4       # Your tuned BM25 parameter

# 6. DATA LOADING
if os.path.exists(QUERIES_PATH):
    queries_df = pd.read_csv(QUERIES_PATH, sep='\t', header=None, names=['qid', 'text'], dtype={'qid': str})
    TRAIN_QIDS = sorted(queries_df['qid'].tolist(), key=int)[:50]
    QUERIES_DICT = dict(zip(queries_df.qid, queries_df.text))
    print(f"✅ Data Ready. Loaded {len(TRAIN_QIDS)} queries.")
else:
    print(f"❌ CRITICAL ERROR: File not found at {QUERIES_PATH}")

##System Initialization

In [ ]:
# --- INITIALIZE GEMINI ---
API_KEY = userdata.get('GOOGLE_API_KEY')
import google.generativeai as genai
genai.configure(api_key=API_KEY)
# We use gemini-3 for maximum reliability in high-volume batch tasks
model = genai.GenerativeModel('gemini-3-flash-preview')

# --- INITIALIZE SEARCHER ---
searcher = LuceneSearcher.from_prebuilt_index('robust04')
searcher.set_bm25(k1=BEST_K1, b=BEST_B)

print("✅ Gemini 3 Flash and Lucene Index Ready.")

## The "Knowledge Harvest" Logic

In [ ]:
def get_q2d_expansion(query_text, num_terms=15):
    prompt = f"Identify {num_terms} most relevant specific associated entities, synonyms and sub-topics for this 90s news query: '{query_text}'. Comma-separated list only."
    response = model.generate_content(prompt).text
    return [t.strip() for t in response.split(",") if t.strip()]

def get_q2d_snippets(query_text, terms, num_snips=10):
    prompt = f"""Query: {query_text} | Keywords: {', '.join(terms)}
    Task: Write {num_snips} diverse 10-word news fragments from the late 90s.
    Each fragment must focus on different combination of the keywords, use all terms. One per line. No numbers.""" #maybe add that they must be relevant to all the query terms
    response = model.generate_content(prompt).text
    return [s.strip() for s in response.split('\n') if s.strip()]

##The Heavy Data-Gathering Loop

In [ ]:
def load_master():
    if os.path.exists(MASTER_DATA_PATH):
        with open(MASTER_DATA_PATH, 'r') as f: return json.load(f)
    return {}

master_data = load_master()
print(f"🚀 Resuming/Starting Harvest... ({len(master_data)}/50 completed)")

# Change the loop start to take the first 10
for qid in tqdm(TRAIN_QIDS[0:10]): # TODO: change to your group 0-10 10-20 ....
    if qid in master_data: continue # Skip if already on Drive

    txt = QUERIES_DICT[qid]

    try:
        # 1. Expand and Generate Snippets
        terms = get_q2d_expansion(txt)
        snippets = get_q2d_snippets(txt, terms)

        # 2. Get Baseline Top 1000
        base_hits = searcher.search(txt, k=1000)
        base_scores = {h.docid: float(h.score) for h in base_hits}
        top_1000_ids = set(base_scores.keys())

        # 3. Score Snippets (Intersecting with Top 1000)
        snippet_raw_scores = {}
        for snip in snippets:
            snip_hits = searcher.search(snip, k=1000)
            # We ONLY save scores for docs that were in the original BM25 top 1000
            snippet_raw_scores[snip] = {h.docid: float(h.score) for h in snip_hits if h.docid in top_1000_ids}

        # 4. Save Raw Ingredients
        master_data[qid] = {
            'query_text': txt,
            'expanded_terms': terms,
            'snippets': snippets,
            'base_bm25_scores': base_scores,
            'snippet_rerank_scores': snippet_raw_scores
        }

        # Checkpoint to Drive immediately
        with open(MASTER_DATA_PATH, 'w') as f: json.dump(master_data, f)
        time.sleep(3) # Protect API Quota

    except exceptions.TooManyRequests:
        print(f"\n⏳ Rate limit hit at QID {qid}. Saved progress. Run this cell again in a few minutes.")
        break
    except Exception as e:
        print(f"\n❌ Unexpected error on QID {qid}: {e}")
        continue

print(f"\n✅ Harvest Complete. Master file updated at: {MASTER_DATA_PATH}")

## Instant Parameter Play & MAP Evaluation

In [ ]:
from tabulate import tabulate

# --- THE TUNING KNOB ---
LAMBDA = 0.35  # Adjust this value and rerun this cell
# ----------------------

def run_evaluation(data, lambd):
    all_rows = []
    for qid, info in data.items():
        base = info['base_bm25_scores']
        snip_info = info['snippet_rerank_scores']

        # Calculate Average Snippet Score
        avg_snip_scores = {}
        for snip_text, scores_dict in snip_info.items():
            for docid, score in scores_dict.items():
                avg_snip_scores[docid] = avg_snip_scores.get(docid, 0) + score

        num_snips = len(snip_info)
        if num_snips > 0:
            for d in avg_snip_scores: avg_snip_scores[d] /= num_snips

        # Linear Interpolation: (1-L)*Base + (L)*SnippetAvg
        fused = []
        for docid, b_score in base.items():
            s_score = avg_snip_scores.get(docid, 0)
            final = ((1 - lambd) * b_score) + (lambd * s_score)
            fused.append((docid, final))

        # Sort and Format for TrecRun
        fused = sorted(fused, key=lambda x: x[1], reverse=True)
        for rank, (docid, score) in enumerate(fused):
            all_rows.append([qid, "Q0", docid, rank+1, score, f"Q2D_L{lambd}"])

    pd.DataFrame(all_rows).to_csv('q2d_final_run.txt', sep=' ', header=False, index=False)
    return TrecRun('q2d_final_run.txt')

# LOAD DATA AND EVALUATE
master_data = load_master()
if master_data:
    qrels = TrecQrel(QRELS_PATH)
    rerank_run = run_evaluation(master_data, LAMBDA)
    te = TrecEval(rerank_run, qrels)

    # Display results
    print(f"\n📊 PERFORMANCE REPORT (LAMBDA = {LAMBDA})")
    metrics = [
        ["Mean Average Precision (MAP)", te.get_map()],
        ["Precision @ 5", te.get_precision(depth=5)],
        ["Precision @ 10", te.get_precision(depth=10)]
    ]
    print(tabulate(metrics, headers=["Metric", "Value"], tablefmt="fancy_grid"))
else:
    print("⚠️ No data found. Run the Harvest Loop (Cell 4) first!")